© 2026 Mechanome Bio. Proprietary and confidential.

# Run GeneSegNet
repo: https://github.com/BoomStarcuc/GeneSegNet

- Python 3.8, pytorch 1.12.1. Not a package: repo cloned into `../external/GeneSegNet`.
- This notebook runs in the shared kernel (needs spatialdata); GeneSegNet's own scripts run in the
  `genesegnet` env through `conda run`.
- Training data = the ground truths in `../inputs/ground_truths/<dataset>/`, used whole (350x350 px), split 3/1/1 by
  ground truth into `../inputs/GeneSegNet_splits/`. No extra tiling: training already rescales each image to
  `--diam_mean` and takes random rotated/rescaled crops every epoch (`GeneSegNet/core.py`).

## Environment (run once)

In [ ]:
from pathlib import Path

GENESEGNET_REPOSITORY = Path("../external/GeneSegNet").resolve()
!git clone https://github.com/BoomStarcuc/GeneSegNet.git {GENESEGNET_REPOSITORY}
!conda env create -f ../env_files/genesegnet.yml
!conda run --no-capture-output -n genesegnet pip install -r {GENESEGNET_REPOSITORY}/requirements.txt

In [ ]:
import json
import random
import sys

import cv2
import numpy
import pandas
import tifffile
from PIL import Image
from scipy.ndimage import distance_transform_edt

sys.path.append("..")
from utils import seg_utils

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
METHOD = "genesegnet"
BOUNDING_BOX_MICRONS = (4673, 1476, 5036, 1805)  # region to segment after training; None = whole slide

GROUND_TRUTH_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "ground_truths" / DATASET_NAME
SPLITS_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "GeneSegNet_splits" / DATASET_NAME
INFERENCE_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "GeneSegNet_inference" / DATASET_NAME
MODEL_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "GeneSegNet_models" / DATASET_NAME
SIGMA = 7  # gaussian width (pixels) for the transcript heatmaps; repo default

sdata = seg_utils.load_sdata(DATASET_NAME)

## Helpers

In [ ]:
def to_8bit(image):
    # GeneSegNet reads images as 8-bit jpgs
    return (numpy.clip(image / numpy.percentile(image, 99.9), 0, 1) * 255).astype(numpy.uint8)


def gaussian_heatmap(spots, shape, sigma=SIGMA):
    # Same as the repo's gen_pose_target: max over per-spot gaussians, zeroed below 0.01.
    # The max of the gaussians = the gaussian of the distance to the nearest spot.
    if len(spots) == 0:
        return numpy.zeros(shape, dtype=numpy.uint8)
    no_spot = numpy.ones(shape, dtype=bool)
    no_spot[spots["spotY"].astype(int), spots["spotX"].astype(int)] = False
    distance = distance_transform_edt(no_spot)
    heatmap = numpy.exp(-distance ** 2 / (2 * sigma ** 2))
    heatmap[heatmap < 0.01] = 0
    return (heatmap * 255).astype(numpy.uint8)


def spots_in_window(x_origin_pixels, y_origin_pixels, width, height):
    # transcripts inside a window of the full image -> spotX/spotY (pixels, relative to the window) + integer gene id
    points = sdata[seg_utils.TRANSCRIPTS_KEY]
    x_min, y_min = x_origin_pixels * seg_utils.PIXEL_SIZE_MICRONS, y_origin_pixels * seg_utils.PIXEL_SIZE_MICRONS
    x_max, y_max = x_min + width * seg_utils.PIXEL_SIZE_MICRONS, y_min + height * seg_utils.PIXEL_SIZE_MICRONS
    window = points[(points["x"] >= x_min) & (points["x"] < x_max) & (points["y"] >= y_min) & (points["y"] < y_max)].compute()
    return pandas.DataFrame({
        "spotX": (window["x"] / seg_utils.PIXEL_SIZE_MICRONS - x_origin_pixels).astype(int),
        "spotY": (window["y"] / seg_utils.PIXEL_SIZE_MICRONS - y_origin_pixels).astype(int),
        "gene": window[seg_utils.GENE_COLUMN].cat.codes.values,  # GeneSegNet reads the gene column as an int
    })

## Training data from the ground truths
Each `crop-N-x<X>-y<Y>-flu.tif` has its origin (scale-0 pixels of the full image) in its tif description.
Ground truths are left as they are; GeneSegNet copies go to `GeneSegNet_splits`.

In [ ]:
ground_truth_images = sorted(GROUND_TRUTH_DIRECTORY.glob("*-flu.tif"))
random.Random(0).shuffle(ground_truth_images)
splits = {"train": ground_truth_images[:3], "val": ground_truth_images[3:4], "test": ground_truth_images[4:5]}

label_areas = []
for split, image_paths in splits.items():
    for image_path in image_paths:
        sample = image_path.name.replace("-flu.tif", "")
        with tifffile.TiffFile(image_path) as tif:
            description = json.loads(tif.pages[0].description)
            image = tif.asarray()
        label = tifffile.imread(image_path.with_name(f"{sample}-mask.tif"))
        spots = spots_in_window(description["x"], description["y"], description["width"], description["height"])
        spots_in_cells = spots[label[spots["spotY"], spots["spotX"]] != 0]
        label_areas.extend(numpy.unique(label[label > 0], return_counts=True)[1])

        sample_directory = SPLITS_DIRECTORY / split / sample
        for subdirectory in ["images", "labels", "spots", "HeatMaps/HeatMap", "HeatMaps/HeatMap_all"]:
            (sample_directory / subdirectory).mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(sample_directory / "images" / f"{sample}_image.jpg"), to_8bit(image))
        cv2.imwrite(str(sample_directory / "labels" / f"{sample}_label.png"), label.astype(numpy.uint16))
        spots.to_csv(sample_directory / "spots" / f"{sample}_spot.csv", index=False)
        cv2.imwrite(str(sample_directory / "HeatMaps/HeatMap_all" / f"{sample}_gaumap_all.jpg"), gaussian_heatmap(spots, label.shape))
        cv2.imwrite(str(sample_directory / "HeatMaps/HeatMap" / f"{sample}_gaumap.jpg"), gaussian_heatmap(spots_in_cells, label.shape))

# cellpose-style diameter (median sqrt(area) / (sqrt(pi)/2)), used for --diameter at evaluation
diameter_pixels = numpy.median(numpy.sqrt(label_areas)) / (numpy.pi ** 0.5 / 2)
print(f"median cell diameter in ground truths: {diameter_pixels:.1f} px")

## Train
README command, plus `--diameter` from the ground truths and `--style_on 1` (the sliding-window inference
script hard-codes `style_on = 1`, so the model is trained the same way).

In [ ]:
train_command = " ".join([
    "conda run --no-capture-output -n genesegnet python -u GeneSeg_train.py --use_gpu",
    f"--train_dir {SPLITS_DIRECTORY / 'train'}",
    f"--val_dir {SPLITS_DIRECTORY / 'val'}",
    f"--test_dir {SPLITS_DIRECTORY / 'test'}",
    "--pretrained_model None --save_png --save_each --img_filter _image --mask_filter _label",
    "--all_channels --verbose --metrics --dir_above --style_on 1",
    f"--diameter {diameter_pixels:.1f}",
    f"--save_model_dir {MODEL_DIRECTORY}",
])
!cd {GENESEGNET_REPOSITORY / "GeneSegNet"} && {train_command}

## Inference on the bounding box
`GeneSeg_test.py` only scores the test tiles; `Inference/slidingwindows_gradient.py` is what segments a full-size
image (256 px windows, stitched). Its paths are hard-coded under `__main__`, so its function is called directly.
It expects `images/*.jpg`, `labels/*.png` (unused, but counted, so a blank one is written) and `spots/*.csv`.
The bounding box must be at least 256 px (~55 µm) on each side.

In [ ]:
sdata_crop = seg_utils.crop_sdata(sdata, BOUNDING_BOX_MICRONS)
dapi = seg_utils.get_image(sdata_crop)
origin_x, origin_y = seg_utils.image_origin_pixels(sdata_crop)
spots = spots_in_window(origin_x, origin_y, dapi.shape[1], dapi.shape[0])

for subdirectory in ["images", "labels", "spots", "results"]:
    (INFERENCE_DIRECTORY / subdirectory).mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(INFERENCE_DIRECTORY / "images" / f"{DATASET_NAME}.jpg"), to_8bit(dapi))
cv2.imwrite(str(INFERENCE_DIRECTORY / "labels" / f"{DATASET_NAME}.png"), numpy.zeros(dapi.shape, dtype=numpy.uint16))
spots.to_csv(INFERENCE_DIRECTORY / "spots" / f"{DATASET_NAME}.csv", index=False)

# newest saved model file
model_path = max((path for path in (MODEL_DIRECTORY / "models").iterdir() if path.is_file()), key=lambda path: path.stat().st_mtime)

inference_call = (f"import slidingwindows_gradient as inference; "
                  f"inference.load_data_and_model('{INFERENCE_DIRECTORY}', '{INFERENCE_DIRECTORY / 'results'}', '{model_path}', {SIGMA})")
!cd {GENESEGNET_REPOSITORY / "Inference"} && conda run --no-capture-output -n genesegnet python -c "{inference_call}"

## Save

In [ ]:
mask = numpy.array(Image.open(INFERENCE_DIRECTORY / "results" / f"label_{DATASET_NAME}.png")).astype(numpy.int32)
polygons = seg_utils.mask_to_polygons(mask, origin_pixels=(origin_x, origin_y))
seg_utils.save_segmentation(sdata, polygons, METHOD, DATASET_NAME, BOUNDING_BOX_MICRONS)